## **REFLACX CSV to Generate Bounding Box**

In [ ]:
import os
import csv
import pandas as pd

# Root directory
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\\mimic-eye"

# Destination CSV file
csv_output_path = r"patients_with_reflacx_folder.csv"

# To store results
rows = []

# Loop over all patient folders
for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if os.path.isdir(patient_path) and patient_folder.startswith("patient_"):
        reflacx_path = os.path.join(patient_path, "REFLACX", "main_data")
        metadata_path = os.path.join(patient_path, "REFLACX", "metadata.csv")
        cxr_jpg_path = os.path.join(patient_path, "CXR-JPG")
        cxr_meta_path = os.path.join(cxr_jpg_path, "cxr_meta.csv")

        if os.path.exists(reflacx_path) and os.path.exists(cxr_meta_path) and os.path.exists(metadata_path):
            try:
                cxr_df = pd.read_csv(cxr_meta_path)
                metadata_df = pd.read_csv(metadata_path)

                if {'study_id', 'dicom_id'}.issubset(cxr_df.columns) and {'dicom_id', 'id'}.issubset(metadata_df.columns):
                    for _, row in cxr_df.iterrows():
                        study_id = row['study_id']
                        dicom_id = row['dicom_id']
                        study_folder_path = os.path.join(cxr_jpg_path, f"s{study_id}")
                        image_path = os.path.join(study_folder_path, f"{dicom_id}.jpg")

                        # Match dicom_id in metadata
                        matched = metadata_df[metadata_df['dicom_id'] == dicom_id]
                        reflacx_id = matched['id'].values[0] if not matched.empty else ""

                        if os.path.exists(image_path):
                            rows.append({
                                "patient_id": patient_folder,
                                "study_id": study_id,
                                "dicom_id": dicom_id,
                                "reflacx_id": reflacx_id,
                                "image_path": image_path,
                                "reflacx_main_folder_path": reflacx_path
                                
                            })
                        else:
                            print(f"Image missing: {image_path}")
                else:
                    print(f"Missing columns in {cxr_meta_path} or {metadata_path}")
            except Exception as e:
                print(f"Error processing {patient_folder}: {e}")

# Write to CSV
with open(csv_output_path, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=[
        "patient_id", "study_id", "dicom_id", "image_path",
        "reflacx_main_folder_path", "reflacx_id"
    ])
    writer.writeheader()
    writer.writerows(rows)

print(f" Successfully saved {len(rows)} rows into '{csv_output_path}'")

✅ Successfully saved 2651 rows into 'patients_with_reflacx_folder.csv'


## **REFLACX Bounding Box Generate**

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import os

# --- CONFIG ---
reflacx_csv = r"patients_with_reflacx_folder.csv"
weights_path = r"best.pt"

# --- Load Model ---
model = YOLO(weights_path)

# --- Load Patient Image Metadata ---
df = pd.read_csv(reflacx_csv)

# --- Storage ---
patient_bboxes = {}

# --- Predict per Image ---
for idx, row in df.iterrows():
    patient_id = row['patient_id']
    dicom_id = row['dicom_id']
    study_id = row['study_id']
    image_path = row['image_path']
    reflacx_folder = os.path.join(Path(image_path).parents[2], "REFLACX")

    try:
        preds = model.predict(
            source=image_path,
            save=False,
            save_txt=False,
            conf=0.05,
            iou=0.5,
            device=0,
            verbose=False
        )[0]

        # Reset best boxes for this image
        best_per_class = {}

        for box in preds.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if cls_id not in best_per_class or best_per_class[cls_id]["confidence"] < conf:
                best_per_class[cls_id] = {
                    "study_id": study_id,
                    "dicom_id": dicom_id,
                    "class_id": cls_id,
                    "class_name": model.names[cls_id],
                    "confidence": conf,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                }

        if patient_id not in patient_bboxes:
            patient_bboxes[patient_id] = {
                "reflacx_folder": reflacx_folder,
                "entries": []
            }

        patient_bboxes[patient_id]["entries"].extend(best_per_class.values())
        print(f"{dicom_id}: {len(best_per_class)} unique class boxes saved")

    except Exception as e:
        print(f"Error on {image_path}: {e}")

# --- Save per-patient CSVs with safe checks ---
# --- Save bounding boxes per dicom_id inside each reflacx_id folder ---
for patient_id, data in patient_bboxes.items():
    df_all = pd.DataFrame(data["entries"])
    if df_all.empty or 'confidence' not in df_all.columns:
        print(f"Skipped empty or invalid data for patient: {patient_id}")
        continue

    # Load patient-specific reflacx_id mapping
    patient_csv_path = os.path.join(root_path, patient_id, "REFLACX", "metadata.csv")
    try:
        reflacx_meta = pd.read_csv(patient_csv_path)
    except Exception as e:
        print(f" Couldn't read metadata for {patient_id}: {e}")
        continue

    reflacx_id_map = dict(zip(reflacx_meta["dicom_id"], reflacx_meta["id"]))

    # Save per dicom_id inside main_data/<reflacx_id>/ directory
    for dicom_id, df_img in df_all.groupby("dicom_id"):
        reflacx_id = reflacx_id_map.get(dicom_id)
        if not reflacx_id:
            print(f" No reflacx_id found for {dicom_id} in {patient_id}")
            continue

        main_data_folder = os.path.join(root_path, patient_id, "REFLACX", "main_data", reflacx_id)
        os.makedirs(main_data_folder, exist_ok=True)

        # Keep only highest-confidence box per class
        df_best = (
            df_img
            .sort_values("confidence", ascending=False)
            .drop_duplicates(subset=["class_id"], keep="first")
            .sort_values(by=["class_id"])
        )

        img_csv = os.path.join(main_data_folder, f"{dicom_id}_bboxes.csv")
        df_best.to_csv(img_csv, index=False)
        print(f"Saved {len(df_best)} boxes for {dicom_id} → {img_csv}")




print("\nAll patient CSVs now safely filtered with unique [dicom_id, class_id].")

dcdc4bd9-4301b111-2a65a814-ee8e7bc5-7f0b9a5a: 1 unique class boxes saved
5bdabba9-388f6646-ac06b5f5-f68b2fd2-3630de21: 5 unique class boxes saved
4a629500-9c3281ca-90bab490-9b6ac9c1-e5e6a580: 0 unique class boxes saved
c48dc8db-35b347c7-deafe540-9d161fdd-7d88173a: 14 unique class boxes saved
38a5ffc8-93297f03-3f0a14e4-aa8ca225-7e968b5b: 17 unique class boxes saved
014cf120-d578617b-edc3682a-28dc6244-d719eda5: 2 unique class boxes saved
fbae5604-a1013966-9879d4a3-7298599f-956824e1: 1 unique class boxes saved
8ccf0b84-84013d2f-287bdfe0-510bfd7b-865dd733: 8 unique class boxes saved
f7b69ee3-db7f264c-fca7d1c7-1d372fc0-02b35a47: 17 unique class boxes saved
d98be0d5-a648e485-4473a3ef-7762b3ef-2d70507e: 10 unique class boxes saved
56b4e225-4dfb070d-783280b4-da33f762-168aa398: 1 unique class boxes saved
6c79a3c1-8d01e0d4-b8b42695-900cae5d-18a8e87a: 17 unique class boxes saved
abea5eb9-b7c32823-3a14c5ca-77868030-69c83139: 11 unique class boxes saved
e5ba5704-ce2f09d3-e28fe2a2-8a9aca96-86f4966a:

## **EyeGaze norm_fixation**

In [3]:
import os
import pandas as pd
import numpy as np

# Root MIMIC-Eye path
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye"

for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if not (os.path.isdir(patient_path) and patient_folder.startswith("patient_")):
        continue

    eyegaze_path = os.path.join(patient_path, "EyeGaze")
    cxr_meta_path = os.path.join(patient_path, "CXR-JPG", "cxr_meta.csv")
    fixations_path = os.path.join(eyegaze_path, "fixations.csv")
    master_sheet_path = os.path.join(eyegaze_path, "master_sheet.csv")

    if not (os.path.exists(eyegaze_path) and os.path.exists(cxr_meta_path) and os.path.exists(fixations_path) and os.path.exists(master_sheet_path)):
        continue

    try:
        cxr_df = pd.read_csv(cxr_meta_path)
        master_df = pd.read_csv(master_sheet_path)
        fix_df = pd.read_csv(fixations_path)

        required_meta_cols = {'study_id', 'dicom_id'}
        required_master_cols = {'dicom_id'}
        required_fix_cols = {'DICOM_ID', 'FPOGX', 'FPOGY', 'FPOGD', 'LPD', 'RPD', 'LPV', 'RPV', 'Time (in secs)'}

        if not (required_meta_cols.issubset(cxr_df.columns) and
                required_master_cols.issubset(master_df.columns) and
                required_fix_cols.issubset(fix_df.columns)):
            print(f"Missing required columns in {patient_folder}")
            continue

        # Find common dicom_ids between master_sheet and cxr_meta
        common_dicom_ids = set(master_df['dicom_id']).intersection(set(cxr_df['dicom_id']))

        # Compute baseline pupil area from first 2 seconds of valid data
        valid_baseline = fix_df[(fix_df['LPV'] == 1) & (fix_df['RPV'] == 1)]
        baseline_window = valid_baseline[valid_baseline['Time (in secs)'] <= 2.0]

        if not baseline_window.empty:
            left_area = np.pi * (baseline_window['LPD'] / 2) ** 2
            right_area = np.pi * (baseline_window['RPD'] / 2) ** 2
            avg_area = (left_area + right_area) / 2
            baseline_avg_area = avg_area.mean()
        else:
            baseline_avg_area = None

        for dicom_id in common_dicom_ids:
            matches = fix_df[fix_df['DICOM_ID'] == dicom_id]
            if matches.empty:
                continue

            # Get study_id for current dicom
            study_row = cxr_df[cxr_df['dicom_id'] == dicom_id]
            study_id = study_row['study_id'].values[0] if not study_row.empty else None

            rows = []
            for _, fix in matches.iterrows():
                pupil_area_norm = ""
                if baseline_avg_area and fix['LPD'] > 0 and fix['RPD'] > 0:
                    left_area = np.pi * (fix['LPD'] / 2) ** 2
                    right_area = np.pi * (fix['RPD'] / 2) ** 2
                    pupil_avg = (left_area + right_area) / 2
                    pupil_area_norm = pupil_avg / baseline_avg_area

                rows.append({
                    "study_id": study_id,
                    "dicom_id": dicom_id,
                    "x_norm": fix["FPOGX"],
                    "y_norm": fix["FPOGY"],
                    "gaze_duration": fix["FPOGD"],
                    "pupil_area_normalized": pupil_area_norm
                })

            if rows:
                output_csv_path = os.path.join(eyegaze_path, f"norm_fixations_{dicom_id}.csv")
                pd.DataFrame(rows).to_csv(output_csv_path, index=False)
                print(f"Saved {len(rows)} rows to {output_csv_path}")

    except Exception as e:
        print(f"Error in {patient_folder}: {e}")

Saved 43 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10002430\EyeGaze\norm_fixations_4873aa08-977bfd31-fb492e64-6ef432d1-3f12cbe3.csv
Saved 29 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012498\EyeGaze\norm_fixations_8106217e-c41ab813-c6002d3f-ed4ee98b-7b165bae.csv
Saved 35 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10015701\EyeGaze\norm_fixations_5f1ac54d-47dfae00-930fa704-514131fa-ee0c138e.csv
Saved 41 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10018423\EyeGaze\norm_fixations_52f7cf0a-e13e2328-993ae39b-37ed0efb-14d64e8a.csv
Saved 38 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimod

## **REFLACX norm_fixation**

In [4]:
import os
import pandas as pd

# Root dataset path
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye"

for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if not patient_folder.startswith("patient_"):
        continue

    reflacx_meta_path = os.path.join(patient_path, "REFLACX", "metadata.csv")
    cxr_meta_path = os.path.join(patient_path, "CXR-JPG", "cxr_meta.csv")
    reflacx_main_data_path = os.path.join(patient_path, "REFLACX", "main_data")

    if not os.path.exists(reflacx_meta_path) or not os.path.exists(cxr_meta_path) or not os.path.exists(reflacx_main_data_path):
        continue

    try:
        reflacx_meta = pd.read_csv(reflacx_meta_path)
        cxr_meta = pd.read_csv(cxr_meta_path)

        if not {'dicom_id', 'id'}.issubset(reflacx_meta.columns):
            print(f"Missing required columns in metadata for {patient_folder}")
            continue

        if not {'study_id', 'dicom_id'}.issubset(cxr_meta.columns):
            print(f"Missing required columns in cxr_meta.csv for {patient_folder}")
            continue

        # Get common dicom_ids
        common_dicom_ids = set(reflacx_meta['dicom_id']).intersection(set(cxr_meta['dicom_id']))

        for reflacx_id in os.listdir(reflacx_main_data_path):
            fix_path = os.path.join(reflacx_main_data_path, reflacx_id, "fixations.csv")
            if not os.path.exists(fix_path):
                continue

            fix_df = pd.read_csv(fix_path)

            required_cols = {
                'x_position', 'y_position',
                'xmin_shown_from_image', 'xmax_shown_from_image',
                'ymin_shown_from_image', 'ymax_shown_from_image',
                'pupil_area_normalized',
                'timestamp_start_fixation', 'timestamp_end_fixation'
            }

            if not required_cols.issubset(fix_df.columns):
                print(f"Required columns missing in {fix_path}")
                continue

            # Match reflacx_id to dicom_id
            dicom_id_matches = reflacx_meta[reflacx_meta['id'] == reflacx_id]
            if dicom_id_matches.empty:
                continue

            dicom_id = dicom_id_matches['dicom_id'].values[0]
            if dicom_id not in common_dicom_ids:
                continue

            # Get study_id from cxr_meta
            study_id_row = cxr_meta[cxr_meta['dicom_id'] == dicom_id]
            study_id = study_id_row['study_id'].values[0] if not study_id_row.empty else None

            # Normalize and compute
            fix_df['x_norm'] = (
                (fix_df['x_position'] - fix_df['xmin_shown_from_image']) /
                (fix_df['xmax_shown_from_image'] - fix_df['xmin_shown_from_image'])
            ).clip(0, 1)

            fix_df['y_norm'] = (
                (fix_df['y_position'] - fix_df['ymin_shown_from_image']) /
                (fix_df['ymax_shown_from_image'] - fix_df['ymin_shown_from_image'])
            ).clip(0, 1)

            fix_df['gaze_duration'] = (
                fix_df['timestamp_end_fixation'] - fix_df['timestamp_start_fixation']
            )

            rows = []
            for _, fix in fix_df.iterrows():
                rows.append({
                    "study_id": study_id,
                    "dicom_id": dicom_id,
                    "reflacx_id": reflacx_id,
                    "x_norm": fix["x_norm"],
                    "y_norm": fix["y_norm"],
                    "gaze_duration": fix["gaze_duration"],
                    "pupil_area_normalized": fix["pupil_area_normalized"]
                })

            # Save inside each reflacx_id folder
            if rows:
                output_path = os.path.join(reflacx_main_data_path, reflacx_id, f"norm_fixations_{dicom_id}.csv")
                pd.DataFrame(rows).to_csv(output_path, index=False)
                print(f"Saved {len(rows)} rows to {output_path}")

    except Exception as e:
        print(f"Error in {patient_folder}: {e}")


Saved 55 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10003400\REFLACX\main_data\P300R570457\norm_fixations_dcdc4bd9-4301b111-2a65a814-ee8e7bc5-7f0b9a5a.csv
Saved 92 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10011607\REFLACX\main_data\P300R471381\norm_fixations_5bdabba9-388f6646-ac06b5f5-f68b2fd2-3630de21.csv
Saved 80 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012261\REFLACX\main_data\P300R911914\norm_fixations_c48dc8db-35b347c7-deafe540-9d161fdd-7d88173a.csv
Saved 128 rows to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012261\REFLACX\main_data\P300R948644\norm_fixations_4a629500-9c3281ca-90bab490-9b6ac9c1-e5e6a580.csv
Sav

## **EyeGaze Bounding Box Separator Code according to dicom_id**

In [5]:
import os
import pandas as pd
from pathlib import Path

# Root of your MIMIC-Eye dataset
root_dir = Path(
    r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-"
    r"for-multimodal-deep-learning-applications-1.0.0\mimic-eye"
)

for patient_folder in root_dir.iterdir():
    # Only patient directories
    if not patient_folder.is_dir() or not patient_folder.name.startswith("patient_"):
        continue

    eye_dir = patient_folder / "EyeGaze"
    bb_csv = eye_dir / "bounding_boxes.csv"
    if not bb_csv.exists():
        continue

    # Load and clean up
    df = pd.read_csv(bb_csv)
    # If there’s an unnamed index column, drop it
    df = df.loc[:, df.columns.str.strip().astype(bool)]

    # For each dicom_id, write a separate CSV
    for dicom_id, group in df.groupby("dicom_id"):
        out_path = eye_dir / f"bounding_boxes_{dicom_id}.csv"
        group[["dicom_id", "bbox_name", "x1", "x2", "y1", "y2"]] \
            .to_csv(out_path, index=False)
        print(f"Wrote {out_path}")

Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10002430\EyeGaze\bounding_boxes_4873aa08-977bfd31-fb492e64-6ef432d1-3f12cbe3.csv
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012498\EyeGaze\bounding_boxes_8106217e-c41ab813-c6002d3f-ed4ee98b-7b165bae.csv
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10015701\EyeGaze\bounding_boxes_5f1ac54d-47dfae00-930fa704-514131fa-ee0c138e.csv
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10018423\EyeGaze\bounding_boxes_52f7cf0a-e13e2328-993ae39b-37ed0efb-14d64e8a.csv
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_1

## **REFLACX transcript.json**

In [6]:
import os
import pandas as pd
import json

# — adjust this to point at the top of your mimic-eye folder —
ROOT = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-" \
       r"for-multimodal-deep-learning-applications-1.0.0\mimic-eye"

for patient in os.listdir(ROOT):
    patient_dir = os.path.join(ROOT, patient)
    if not (patient.startswith("patient_") and os.path.isdir(patient_dir)):
        continue

    main_data = os.path.join(patient_dir, "REFLACX", "main_data")
    if not os.path.isdir(main_data):
        continue

    for reflacx_id in os.listdir(main_data):
        folder = os.path.join(main_data, reflacx_id)
        csv_path = os.path.join(folder, "timestamps_transcription.csv")
        if not os.path.exists(csv_path):
            continue

        # load the CSV
        df = pd.read_csv(csv_path)

        # build the full_text by joining all words in order
        full_text = " ".join(df["word"].astype(str).tolist())

        # build the timed phrases list
        time_stamped = []
        for _, row in df.iterrows():
            time_stamped.append({
                "begin_time": float(row["timestamp_start_word"]),
                "end_time":   float(row["timestamp_end_word"]),
                "phrase":     row["word"]
            })

        # assemble JSON structure
        transcript = {
            "full_text": full_text,
            "time_stamped_text": time_stamped
        }

        # write out transcript.json
        out_path = os.path.join(folder, "transcript.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(transcript, f, ensure_ascii=False, indent=4)

        print(f"Wrote {out_path}")

Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10002428\REFLACX\main_data\P300R081391\transcript.json
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10003400\REFLACX\main_data\P300R570457\transcript.json
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10011607\REFLACX\main_data\P300R471381\transcript.json
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012261\REFLACX\main_data\P300R911914\transcript.json
Wrote mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\patient_10012261\REFLACX\main_data\P300R948644\transcript.json
Wrote mimic-eye-integrating-mimic-datasets-with-re

## **Boss Code -> Gathers everything in a single file**

In [ ]:
import os
import pandas as pd

# Root path
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-" \
            r"multimodal-deep-learning-applications-1.0.0\mimic-eye"

rows = []

def get_report_path(patient_path, study_id):
    if pd.isna(study_id):
        return None
    sid = str(int(study_id)) if isinstance(study_id, (int, float)) else str(study_id)
    p = os.path.join(patient_path, "CXR-DICOM", f"s{sid}.txt")
    return p if os.path.exists(p) else None

def get_image_path(patient_path, study_id, dicom_id):
    if pd.isna(study_id) or pd.isna(dicom_id):
        return None
    sid = str(int(study_id)) if isinstance(study_id, (int, float)) else str(study_id)
    did = str(dicom_id)
    p = os.path.join(patient_path, "CXR-JPG", f"s{sid}", f"{did}.jpg")
    return p if os.path.exists(p) else None

def get_fixations_path(patient_path, source, dicom_id, reflax_id=None):
    if pd.isna(dicom_id):
        return None
    did = str(dicom_id)
    if source == 'eyegaze':
        p = os.path.join(patient_path, "EyeGaze", f"norm_fixations_{did}.csv")
    else:
        if reflax_id is None:
            return None
        rid = str(reflax_id)
        p = os.path.join(patient_path, "REFLACX", "main_data", rid, f"norm_fixations_{did}.csv")
    return p if os.path.exists(p) else None

def get_bbox_path(patient_path, source, dicom_id, reflax_id=None):
    if pd.isna(dicom_id):
        return None
    did = str(dicom_id)
    if source == 'eyegaze':
        p = os.path.join(patient_path, "EyeGaze", f"bounding_boxes_{did}.csv")
    else:
        if reflax_id is None:
            return None
        rid = str(reflax_id)
        p = os.path.join(patient_path, "REFLACX", "main_data", rid, f"{did}_bboxes.csv")
    return p if os.path.exists(p) else None

def get_transcript_path(patient_path, source, dicom_id, reflax_id=None):
    if pd.isna(dicom_id):
        return None
    did = str(dicom_id)
    if source == 'eyegaze':
        p = os.path.join(
            patient_path, "EyeGaze", "audio_segmentation_transcripts", did, "transcript.json"
        )
    else:
        if reflax_id is None:
            return None
        rid = str(reflax_id)
        p = os.path.join(patient_path, "REFLACX", "main_data", rid, "transcript.json")
    return p if os.path.exists(p) else None

for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)
    if not (os.path.isdir(patient_path) and patient_folder.startswith("patient_")):
        continue

    has_reflacx = os.path.isdir(os.path.join(patient_path, "REFLACX"))
    has_eyegaze = os.path.isdir(os.path.join(patient_path, "EyeGaze"))

    # --- load chexpert conditions ---
    chex_path = os.path.join(patient_path, "CXR-JPG", "cxr_chexpert.csv")
    cond_map = {}
    if os.path.exists(chex_path):
        chex_df = pd.read_csv(chex_path)
        if 'study_id' in chex_df.columns:
            for _, cr in chex_df.iterrows():
                sid = str(int(cr['study_id'])) if isinstance(cr['study_id'], (int, float)) else str(cr['study_id'])
                # pick cols with value == 1.0
                conds = [
                    col for col in chex_df.columns
                    if col not in ('subject_id','study_id') and cr.get(col) == 1.0
                ]
                cond_map[sid] = '|'.join(conds) if conds else None

    # --- load cxr_meta ---
    cxr_meta_file = os.path.join(patient_path, "CXR-JPG", "cxr_meta.csv")
    if not os.path.exists(cxr_meta_file):
        rows.append({
            "patient_folder": patient_folder,
            "ifReflacx": has_reflacx,
            "ifEyegaze": has_eyegaze,
            "dicom_id": None,
            "study_id": None,
            "reflax_id": None,
            "report_path": None,
            "image_path": None,
            "fixations_path": None,
            "bbox_path": None,
            "transcript_path": None,
            "condition": None
        })
        continue

    try:
        cxr_df = pd.read_csv(cxr_meta_file)
    except Exception as e:
        print(f" Error reading cxr_meta.csv for {patient_folder}: {e}")
        continue
    if not {'dicom_id','study_id'}.issubset(cxr_df.columns):
        print(f" Missing cols in cxr_meta.csv for {patient_folder}")
        continue

    # Helper to look up condition
    def lookup_condition(study_id):
        sid = str(int(study_id)) if isinstance(study_id, (int, float)) else str(study_id)
        return cond_map.get(sid)

    # --- EYEGAZE only ---
    if has_eyegaze and not has_reflacx:
        ms_file = os.path.join(patient_path, "EyeGaze", "master_sheet.csv")
        ms = pd.read_csv(ms_file) if os.path.exists(ms_file) else pd.DataFrame(columns=['dicom_id'])
        keep = set(ms.get('dicom_id', []))
        sub = cxr_df[cxr_df['dicom_id'].isin(keep)]
        for _, r in sub.iterrows():
            rows.append({
                "patient_folder": patient_folder,
                "ifReflacx": False,
                "ifEyegaze": True,
                "dicom_id": r["dicom_id"],
                "study_id": r["study_id"],
                "reflax_id": None,
                "report_path": get_report_path(patient_path, r["study_id"]),
                "image_path": get_image_path(patient_path, r["study_id"], r["dicom_id"]),
                "fixations_path": get_fixations_path(patient_path, 'eyegaze', r["dicom_id"]),
                "bbox_path": get_bbox_path(patient_path, 'eyegaze', r["dicom_id"]),
                "transcript_path": get_transcript_path(patient_path, 'eyegaze', r["dicom_id"]),
                "condition": lookup_condition(r["study_id"])
            })
        continue

    # --- REFLACX only ---
    if has_reflacx and not has_eyegaze:
        meta_file = os.path.join(patient_path, "REFLACX", "metadata.csv")
        rf = pd.read_csv(meta_file) if os.path.exists(meta_file) else pd.DataFrame(columns=['dicom_id','id'])
        if {'dicom_id','id'}.issubset(rf.columns):
            mapping = dict(zip(rf['dicom_id'], rf['id']))
            keep = set(mapping.keys())
            sub = cxr_df[cxr_df['dicom_id'].isin(keep)]
            for _, r in sub.iterrows():
                rid = mapping[r["dicom_id"]]
                rows.append({
                    "patient_folder": patient_folder,
                    "ifReflacx": True,
                    "ifEyegaze": False,
                    "dicom_id": r["dicom_id"],
                    "study_id": r["study_id"],
                    "reflax_id": rid,
                    "report_path": get_report_path(patient_path, r["study_id"]),
                    "image_path": get_image_path(patient_path, r["study_id"], r["dicom_id"]),
                    "fixations_path": get_fixations_path(patient_path, 'reflacx', r["dicom_id"], reflax_id=rid),
                    "bbox_path": get_bbox_path(patient_path, 'reflacx', r["dicom_id"], reflax_id=rid),
                    "transcript_path": get_transcript_path(patient_path, 'reflacx', r["dicom_id"], reflax_id=rid),
                    "condition": lookup_condition(r["study_id"])
                })
        continue

    # --- BOTH Eyegaze & Reflacx ---
    if has_eyegaze and has_reflacx:
        ms_df = pd.read_csv(os.path.join(patient_path, "EyeGaze", "master_sheet.csv")) \
                if os.path.exists(os.path.join(patient_path, "EyeGaze", "master_sheet.csv")) \
                else pd.DataFrame(columns=['dicom_id'])
        rf_df = pd.read_csv(os.path.join(patient_path, "REFLACX", "metadata.csv")) \
                if os.path.exists(os.path.join(patient_path, "REFLACX", "metadata.csv")) \
                else pd.DataFrame(columns=['dicom_id','id'])

        cnt_e = ms_df['dicom_id'].nunique() if 'dicom_id' in ms_df.columns else 0
        cnt_r = rf_df['dicom_id'].nunique() if 'dicom_id' in rf_df.columns else 0

        if cnt_e >= cnt_r:
            source, keep = 'eyegaze', set(ms_df.get('dicom_id', []))
        else:
            source, keep = 'reflacx', set(rf_df.get('dicom_id', []))

        mapping = dict(zip(rf_df['dicom_id'], rf_df['id'])) if {'dicom_id','id'}.issubset(rf_df.columns) else {}

        sub = cxr_df[cxr_df['dicom_id'].isin(keep)]
        for _, r in sub.iterrows():
            rid = mapping.get(r["dicom_id"])
            rows.append({
                "patient_folder": patient_folder,
                "ifReflacx": True,
                "ifEyegaze": True,
                "dicom_id": r["dicom_id"],
                "study_id": r["study_id"],
                "reflax_id": rid,
                "report_path": get_report_path(patient_path, r["study_id"]),
                "image_path": get_image_path(patient_path, r["study_id"], r["dicom_id"]),
                "fixations_path": get_fixations_path(patient_path, source, r["dicom_id"], reflax_id=rid),
                "bbox_path": get_bbox_path(patient_path, source, r["dicom_id"], reflax_id=rid),
                "transcript_path": get_transcript_path(patient_path, source, r["dicom_id"], reflax_id=rid),
                "condition": lookup_condition(r["study_id"])
            })
        continue

    # --- NEITHER ---
    for _, r in cxr_df.iterrows():
        rows.append({
            "patient_folder": patient_folder,
            "ifReflacx": has_reflacx,
            "ifEyegaze": has_eyegaze,
            "dicom_id": r["dicom_id"],
            "study_id": r["study_id"],
            "reflax_id": None,
            "report_path": get_report_path(patient_path, r["study_id"]),
            "image_path": get_image_path(patient_path, r["study_id"], r["dicom_id"]),
            "fixations_path": None,
            "bbox_path": None,
            "transcript_path": None,
            "condition": lookup_condition(r["study_id"])
        })

# Write out
df = pd.DataFrame(rows)
output_path = os.path.join(root_path, "..", "final_dataset.csv")
df.to_csv(output_path, index=False)
print(f" final_dataset.csv written to {output_path}")


✅ final_dataset.csv written to mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye\..\final_dataset.csv
